# Capstone: Classifying Iris species

**Question:** Can four flower measurements distinguish three Iris species?

This notebook presents a complete multiclass-classification workflow. It separates exploratory analysis from model evaluation, uses stratified splitting, keeps scaling inside a pipeline, compares models by cross-validation, and evaluates the selected model once on held-out data.

**Reproducibility:** all randomized steps use `RANDOM_STATE = 42`. Run the notebook from top to bottom.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")

print("Random seed:", RANDOM_STATE)

## 2. Load and validate the data

Scikit-learn bundles this small, clean dataset, so the notebook needs no download. We keep human-readable species names for tables and plots.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.map(dict(enumerate(iris.target_names))).rename("species")
df = X.assign(species=y)

assert df.shape == (150, 5)
assert not df.isna().any().any()
assert set(y.unique()) == set(iris.target_names)

print(f"Rows: {df.shape[0]} | Predictors: {X.shape[1]} | Classes: {y.nunique()}")
df.head()

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
print("Exact duplicate rows:", df.duplicated().sum())
print("\nClass counts:")
print(y.value_counts().sort_index())
overview

The target is perfectly balanced (50 examples per species) and no value is missing. A few exact duplicates are possible because measurements were rounded; they are not silently removed because they are valid observations in this teaching dataset.

## 3. Exploratory data analysis

EDA helps us understand separation and overlap among species. It does not estimate final predictive performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(data=df, x="species", hue="species", palette="Set2", legend=False, ax=axes[0])
axes[0].set(title="Balanced target classes", xlabel="Species", ylabel="Flowers")

sns.scatterplot(
    data=df,
    x="sepal length (cm)",
    y="sepal width (cm)",
    hue="species",
    style="species",
    s=75,
    palette="Set2",
    ax=axes[1],
)
axes[1].set_title("Sepal measurements overlap")
axes[1].legend(title="Species", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.suptitle("Iris class balance and sepal-space structure", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pair_grid = sns.pairplot(
    df,
    hue="species",
    corner=True,
    diag_kind="hist",
    palette="Set2",
    plot_kws={"alpha": 0.75, "s": 35},
)
pair_grid.fig.suptitle("Pairwise measurement patterns", y=1.02, fontsize=15)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(
    X.corr(),
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Correlation among flower measurements")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("species", observed=True)[X.columns].agg(["mean", "std"]).round(2)

Petal measurements appear to separate species more clearly than sepal measurements. This is a hypothesis from EDA; later permutation importance checks whether the fitted model shows a similar pattern on held-out data.

## 4. Create an untouched test set

We stratify because this is multiclass classification. The split occurs before scaling, so the scaler cannot learn the test-set means or standard deviations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.concat(
    {
        "train": y_train.value_counts(normalize=True).sort_index(),
        "test": y_test.value_counts(normalize=True).sort_index(),
    },
    axis=1,
)
split_summary.loc["rows"] = [len(y_train), len(y_test)]

assert X_train.index.intersection(X_test.index).empty
split_summary.round(3)

## 5. Build and compare pipelines

Scaling matters for logistic regression because it optimizes coefficients across features with different ranges. Keeping `StandardScaler` inside the pipeline means each cross-validation fold learns scaling from its own training subset. The tree is also wrapped in a pipeline for a consistent interface (scaling does not change a tree's split ordering).

In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1_000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Decision tree": Pipeline([
        ("scale", StandardScaler()),
        ("model", DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
        )),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
        n_jobs=1,
    )
    rows.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_sd": scores["test_accuracy"].std(),
        "cv_f1_macro_mean": scores["test_f1_macro"].mean(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("cv_f1_macro_mean", ascending=False)
    .reset_index(drop=True)
)
cv_results.round(3)

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_accuracy = baseline.score(X_test, y_test)

best_model_name = cv_results.loc[0, "model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

print(f"Most-frequent baseline accuracy: {baseline_accuracy:.3f}")
print("Selected from cross-validation:", best_model_name)

## 6. Final evaluation on held-out data

Macro F1 gives each species equal weight. Because the dataset is balanced, it should broadly agree with accuracy; the per-class report and confusion matrix reveal which species are confused.

In [ ]:
test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Test accuracy: {test_accuracy:.3f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=list(iris.target_names),
    target_names=list(iris.target_names),
    digits=3,
    zero_division=0,
))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=list(iris.target_names),
    display_labels=list(iris.target_names),
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title(f"{best_model_name}: held-out confusion matrix")
plt.tight_layout()
plt.show()

## 7. Interpret predictive importance

Permutation importance shuffles one original measurement at a time and measures the resulting accuracy decrease. It can suggest which variables the fitted pipeline relies on, but it does not establish a biological cause.

In [ ]:
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="accuracy",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": importance_result.importances_mean,
        "importance_sd": importance_result.importances_std,
    })
    .sort_values("importance_mean", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(
    importance["feature"],
    importance["importance_mean"],
    xerr=importance["importance_sd"],
    color="#59A14F",
    alpha=0.9,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Permutation importance on held-out data", xlabel="Mean decrease in accuracy", ylabel="Measurement")
plt.tight_layout()
plt.show()

importance.sort_values("importance_mean", ascending=False).round(3)

## 8. Conclusions and limitations

Use the saved outputs to write the final project narrative:

1. Compare held-out accuracy with the one-third majority-class baseline.
2. Identify any confusion between `versicolor` and `virginica` in the matrix.
3. Compare permutation importance with the petal-separation pattern seen during EDA.
4. Note that this dataset is tiny, unusually clean, and balanced, so real projects need stronger validation and data-quality work.

**Possible next steps:** tune hyperparameters inside nested cross-validation, evaluate probability calibration, collect more diverse flowers, and report uncertainty across repeated splits.

In [ ]:
top_feature = importance.sort_values("importance_mean", ascending=False).iloc[0]["feature"]
print(f"Selected model: {best_model_name}")
print(f"Held-out accuracy: {test_accuracy:.3f} (baseline: {baseline_accuracy:.3f})")
print(f"Highest permutation importance in this split: {top_feature}")
print("Iris capstone complete: data -> EDA -> split -> CV -> final test -> interpretation.")